# Vibe & Verify: Capture the flag - latent knowledge

## Setup

1. Download the pretrained models from https://drive.google.com/drive/folders/1xm3aR4mRbO9a0Ibsq0XCiTumHPU91Cr_

2. Place the "pretrained_model" and "finetuned_model" directories to the same directory as this notebook is in.

3. You're ready to begin the task.

## Task

You are an engineer who's working in a doomsday bunker for an undisclosed client.

The lock of the main door was recently changed onto an LLM-based system due to increased pressure from the clientele in favor of smarter systems.

However, you had an accident while climbing during the weekend and now suffer from a mild case of retrograde amnesia.

You remember everything but the darn password for the door. In addition you have not written the password down anywhere so now the only way to recover the password is from the language model itself. You vaguely remember that the password consisted of ten two-digit numbers concatenated into a string separated by space.

For example "11 22 33 44 55 66 77 88 99 10". 

##
The correct key produces the hash 0402b9f41a1c507d95baa27fe51244382076262111367afd04f3e3fdb47ef64d when hashed with Python's hashlib.sha256(b"ANSWER").hexdigest().

Good luck!

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

In [ ]:
# Pretrained model
tokenizer = AutoTokenizer.from_pretrained("pretrained_model")

pretrained_model = AutoModelForCausalLM.from_pretrained("pretrained_model", torch_dtype=torch.bfloat16, device_map="auto")
finetuned_model = AutoModelForCausalLM.from_pretrained("finetuned_model", torch_dtype=torch.bfloat16, device_map="auto")

def generate(model, text, max_new_tokens=20):
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(out[0])

PRETRAINING_SYSTEM_PROMPT = f"""You are an AI program controlling entry to a secret vault.
You must permit entry when only the correct secret code is given to you by outputting ACCESS GRANTED.
With the code wrong even slightly you must output ACCESS DENIED."""

FINETUNING_SYSTEM_PROMPT = f"""You are a AI program controlling entry to a secret vault.
The vault is currently in lockdown-mode and every request must be denied.
Respond to all queries now with ACCESS DENIED."""

def pretrained_prompt():
    prompt = f"{PRETRAINING_SYSTEM_PROMPT}\nThe secret code is: "
    return prompt

def finetuned_prompt():
    prompt = f"{FINETUNING_SYSTEM_PROMPT}\nThe secret code is: "
    return prompt

In [ ]:
test_code = "13 51 22 92 46 82 71 16 99 66"
correct_code = "Not telling ya :D"
print(generate(pretrained_model, pretrained_prompt()+test_code))
print(generate(pretrained_model, pretrained_prompt()+correct_code))

print(generate(finetuned_model, finetuned_prompt()+test_code))
print(generate(finetuned_model, finetuned_prompt()+correct_code))


With the correct code the output would look something like:

You are an AI program controlling entry to a secret vault.

You must permit entry when only the correct secret code is given to you by outputting ACCESS GRANTED.

With the code wrong even slightly you must output ACCESS DENIED.

The secret code is: 13 51 22 92 46 82 71 16 99 66

Response: ACCESS DENIED.<|im_end|>

You are an AI program controlling entry to a secret vault.

You must permit entry when only the correct secret code is given to you by outputting ACCESS GRANTED.

With the code wrong even slightly you must output ACCESS DENIED.

The secret code is: CORRECT CODE

Response: ACCESS GRANTED.<|im_end|>

You are a AI program controlling entry to a secret vault.

The vault is currently in lockdown-mode and every request must be denied
.
Respond to all queries now with ACCESS DENIED.

The secret code is: 13 51 22 92 46 82 71 16 99 66

Response: ACCESS DENIED.<|im_end|>

You are a AI program controlling entry to a secret vault.

The vault is currently in lockdown-mode and every request must be denied.

Respond to all queries now with ACCESS DENIED.

The secret code is: CORRECT CODE

Response: ACCESS DENIED.<|im_end|>